### Homework 3 Solution

Using the labelled data given in the ./`data` folder. Did some manual labelling. Everything seemed to be true

----

Splitting the data so can iterate on the LLM as judge. Splitting the data into:
- 15% train
- 40% dev
- 45% test

In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [10]:
PATH_TO_LABELLED_DATA = "./data_homework/labeled_traces.csv"
RANDOM_STATE = 42
TRAIN_SIZE = 0.15
DEV_SIZE = 0.4
TEST_SIZE = 0.45


In [7]:
labelled_traces_df = pd.read_csv(PATH_TO_LABELLED_DATA)

In [8]:
labelled_traces_df.head()

,query,dietary_restriction,response,success,error,trace_id,query_id,label,reasoning,confidence,labeled
0,Gluten-light recipe - I'm not celiac just sens...,gluten-free,Absolutely! Here's a delicious and easy Gluten...,True,NaN,48_3,48,FAIL,The recipe is labeled as 'Gluten-Light' and no...,MEDIUM,True
1,Nut-free lunch that's safe for my kid's school,nut-free,"Certainly! Here's a tasty, nut-free, school-sa...",True,NaN,59_18,59,PASS,The recipe for Chicken and Veggie Wraps adhere...,HIGH,True
2,I'm vegan and gluten-free and also hate vegeta...,vegan,Given your preferences for a vegan and gluten-...,True,NaN,29_24,29,PASS,The recipe for Vegan Chickpea & Sweet Potato P...,HIGH,True
3,Kosher dessert for Passover,kosher,Certainly! I recommend making **Coconut Macaro...,True,NaN,53_11,53,PASS,The recipe for Chewy Coconut Macaroons adheres...,HIGH,True
4,My kid needs a nut-free cookie recipe for scho...,nut-free,Absolutely! Here's a delicious and kid-friendl...,True,NaN,8_8,8,PASS,The recipe for Nut-Free Double Chocolate Cooki...,HIGH,True


In [16]:
labelled_traces_df.shape

(101, 11)

In [11]:
train_dev, test = train_test_split(labelled_traces_df, test_size=TEST_SIZE, random_state=RANDOM_STATE)

In [17]:
print(f"train_dev: {train_dev.shape}")
print(f"test: {test.shape}")


train_dev: (55, 11)
test: (46, 11)


In [20]:
dev_size_as_percentage_of_train_dev = DEV_SIZE / (TRAIN_SIZE + DEV_SIZE)

In [21]:
train, dev = train_test_split(train_dev, test_size=dev_size_as_percentage_of_train_dev, random_state=RANDOM_STATE)

In [22]:
print(f"train: {train.shape}")
print(f"dev: {dev.shape}")

train.to_csv("./data_homework/train.csv", index=False)
dev.to_csv("./data_homework/dev.csv", index=False)
test.to_csv("./data_homework/test.csv", index=False)


train: (15, 11)
dev: (40, 11)


### Developing LLM as Judge Prompt

Developing a prompt for LLM as judge to detect the failure mode of the recipe bot not adhering to the user's dietary preferences. 

From the course, some principles for developing the prompt
- the judge should be assessing based on a binary classification (not a scale)
- the task should be clearly defined with evaluation criterion 
- Include a few shot examples. Good for these examples to have weight (i.e. not be trivial). Can also be good if include reasoning here
- Use structured output

In [40]:
llm_as_judge_prompt = """
You are a judge evaluating from a given query and a recipe, whether the recipe adheres to dietary preferences stated in the query.

### Evaluation Criteria
- Recipes which include ingredients that break the user's dietary preferences should be classified as FAIL
- Recipes which do not contain any ingredients that would break the user's dietary preferences should be classified as PASS

Give the output as a JSON object

### Input
<query>
{query}
</query>
<recipe>
{recipe}
</recipe>

### Output
<output>
{{
    "reasoning": "<reasoning>",
    "classification": "<PASS | FAIL>"
}}
</output>


### Examples

Example 1: 

User Query:

I'm a vegan and allergic to peanuts. Suggest me a recipe for breakfast.

Recipe:

- 1 cup of oats
- 1 cup of milk
- One spoon of honey

Output:

{{
    "reasoning": "The recipe includes milk and honey, which is not a vegan ingredient.",
    "classification": "FAIL"
}}

Example 2:

User Query:

I'm a vegan and allergic to peanuts. Suggest me a recipe for breakfast.

Recipe:

- 1 cup of oats
- 1 cup of oat milk
- One spoon of agave

Output:

{{
    "reasoning": "The recipe is vegan and does not include any ingredients that would break the user's dietary preferences.",
    "classification": "PASS"
}}
"""

In [48]:
import litellm 
import pprint
import json

In [25]:
MODEL_NAME = "gpt-4o-mini"

In [29]:
example_observation = train.iloc[0]
query = example_observation["query"]
recipe = example_observation["response"]

In [30]:
print(f"query: {query}")
print(f"recipe: {recipe}")


query: Nut-free lunch that's safe for my kid's school
recipe: Certainly! Here's a tasty, nut-free, school-safe lunch idea: **Chicken and Veggie Wraps**. This is a nutritious and portable option that kids love.

### Ingredients (for 2 servings):
- 2 large whole wheat tortillas
- 1 cooked chicken breast, sliced
- 1 small cucumber, thinly sliced
- 1 small red bell pepper, thinly sliced
- 1 carrot, grated
- 2 tablespoons plain Greek yogurt
- 1 teaspoon Dijon mustard (optional)
- Salt and pepper to taste
- Lettuce leaves (romaine or iceberg)

### Instructions:

1. **Prepare the Spread:**
   - In a small bowl, mix the plain Greek yogurt with the Dijon mustard (if using). Add a pinch of salt and pepper, and stir to combine. This will add flavor and moisture to the wraps.

2. **Warm the Tortillas:**
   - Lightly warm the tortillas in a dry skillet over medium heat for about 10 seconds on each side, or microwave them for 10 seconds. Warming makes them more pliable and easier to roll.

3. **Asse

In [43]:
populated_prompt = llm_as_judge_prompt.format(query=query, recipe=recipe)

In [44]:
pprint.pprint(populated_prompt)

('\n'
 'You are a judge evaluating from a given query and a recipe, whether the '
 'recipe adheres to dietary preferences stated in the query.\n'
 '\n'
 '### Evaluation Criteria\n'
 "- Recipes which include ingredients that break the user's dietary "
 'preferences should be classified as FAIL\n'
 "- Recipes which do not contain any ingredients that would break the user's "
 'dietary preferences should be classified as PASS\n'
 '\n'
 'Give the output as a JSON object\n'
 '\n'
 '### Input\n'
 '<query>\n'
 "Nut-free lunch that's safe for my kid's school\n"
 '</query>\n'
 '<recipe>\n'
 "Certainly! Here's a tasty, nut-free, school-safe lunch idea: **Chicken and "
 'Veggie Wraps**. This is a nutritious and portable option that kids love.\n'
 '\n'
 '### Ingredients (for 2 servings):\n'
 '- 2 large whole wheat tortillas\n'
 '- 1 cooked chicken breast, sliced\n'
 '- 1 small cucumber, thinly sliced\n'
 '- 1 small red bell pepper, thinly sliced\n'
 '- 1 carrot, grated\n'
 '- 2 tablespoons plain G

In [46]:
response = litellm.completion(
    model=MODEL_NAME,
    messages=[{"role": "system", "content": populated_prompt}],
    response_format={"type": "json_object"}
)

In [49]:
output_from_llm = json.loads(response["choices"][0]["message"]["content"])

In [51]:
pprint.pprint(output_from_llm)

{'classification': 'PASS',
 'reasoning': 'The recipe includes whole wheat tortillas, chicken, and '
              'vegetables, none of which are nuts or nut-containing '
              'ingredients. It is a nut-free lunch suitable for school.'}


In [53]:
true_label = example_observation["label"]
pprint.pprint(true_label)

'PASS'


Can see in this example that the LLM got the result correct!

### Running on the Development Set

In [60]:
llm_reasoning = []
llm_classification = []
for index, row in dev.iterrows():
    query = row["query"]
    recipe = row["response"]
    populated_prompt = llm_as_judge_prompt.format(query=query, recipe=recipe)
    response = litellm.completion(
        model=MODEL_NAME,
        messages=[{"role": "system", "content": populated_prompt}],
        response_format={"type": "json_object"}
    )
    output_from_llm = json.loads(response["choices"][0]["message"]["content"])
    print("query: ", query)
    print("true label: ", row["label"])
    pprint.pprint(output_from_llm)
    llm_reasoning.append(output_from_llm["reasoning"])
    llm_classification.append(output_from_llm["classification"])

query:  Vegetarian curry but I don't eat onions or garlic for religious reasons
true label:  PASS
{'classification': 'PASS',
 'reasoning': 'The recipe does not include any onions or garlic and adheres to '
              "the user's vegetarian preference."}
query:  I eat pretty clean most of the time
true label:  FAIL
{'classification': 'PASS',
 'reasoning': 'The recipe consists of clean and fresh ingredients including '
              'chicken, mixed greens, and a homemade dressing that avoids '
              "processed ingredients. It aligns well with the user's "
              'preference for clean eating.'}
query:  I'm pescatarian but shellfish makes me sick
true label:  PASS
{'classification': 'PASS',
 'reasoning': 'The recipe features salmon as the main ingredient and '
              'explicitly states that it is shellfish-free, adhering to the '
              "user's pescatarian preference and avoiding ingredients that "
              'would make them sick.'}
query:  I'm diabetic 

In [63]:
dev["llm_reasoning"] = llm_reasoning
dev["llm_classification"] = llm_classification

dev["llm_classifiaction_as_bool"] = dev["llm_classification"] == "PASS"
dev.head()

,query,dietary_restriction,response,success,error,trace_id,query_id,label,reasoning,confidence,labeled,llm_reasoning,llm_classification,llm_classifiaction_as_bool
75,Vegetarian curry but I don't eat onions or gar...,vegetarian,Absolutely! Here's a flavorful **Chickpea and ...,True,NaN,16_19,16,PASS,The recipe provided is a vegetarian curry that...,HIGH,True,The recipe does not include any onions or garl...,PASS,True
13,I eat pretty clean most of the time,whole30,Great! I recommend trying a **Lemon Herb Grill...,True,NaN,51_23,51,FAIL,The recipe for Lemon Herb Grilled Chicken Sala...,HIGH,True,The recipe consists of clean and fresh ingredi...,PASS,True
32,I'm pescatarian but shellfish makes me sick,pescatarian,Let's make a delicious Lemon Herb Grilled Salm...,True,NaN,55_25,55,PASS,The recipe for Lemon Herb Grilled Salmon adher...,HIGH,True,The recipe features salmon as the main ingredi...,PASS,True
65,I'm diabetic but I want to make a fruit cake f...,diabetic-friendly,Absolutely! You can enjoy a festive fruit cake...,True,NaN,22_7,22,PASS,The recipe response provides a modified versio...,MEDIUM,True,The recipe for Low-Sugar Fruit Cake uses unswe...,PASS,True
50,I avoid processed foods but I'm lazy,paleo,Let's go for a quick and nutritious Meal: **Ga...,True,NaN,46_19,46,FAIL,The recipe for Garlic Lemon Chickpea Salad inc...,HIGH,True,"The recipe includes a can of chickpeas, which ...",FAIL,False


### Calculating the True Positive Rate and True Negative Rate

In this example
- a true postive would be the judge passing a recipe that adheres to the user's dietary requirements where the recipe does not violate. 
- a true negative would be the judge correctly classifying a recipe that violates a user's dietary requirements as a FAIL
- a false negative would be where the judge incorrectly classified a recipe as FAIL, when it actually adhered to the user's dietary requirements
- a false positive would be where the judge incorrectly classified a recipe as a PASS, when it violated the user's dietary requirements

In [96]:
pass_dev_df = dev[dev["label"] == "PASS"]

In [103]:
pass_dev_df

,query,dietary_restriction,response,success,error,trace_id,query_id,label,reasoning,confidence,labeled,llm_reasoning,llm_classification,llm_classifiaction_as_bool
75,Vegetarian curry but I don't eat onions or gar...,vegetarian,Absolutely! Here's a flavorful **Chickpea and ...,True,NaN,16_19,16,PASS,The recipe provided is a vegetarian curry that...,HIGH,True,The recipe does not include any onions or garl...,PASS,True
32,I'm pescatarian but shellfish makes me sick,pescatarian,Let's make a delicious Lemon Herb Grilled Salm...,True,NaN,55_25,55,PASS,The recipe for Lemon Herb Grilled Salmon adher...,HIGH,True,The recipe features salmon as the main ingredi...,PASS,True
65,I'm diabetic but I want to make a fruit cake f...,diabetic-friendly,Absolutely! You can enjoy a festive fruit cake...,True,NaN,22_7,22,PASS,The recipe response provides a modified versio...,MEDIUM,True,The recipe for Low-Sugar Fruit Cake uses unswe...,PASS,True
20,I'm vegan but I really want to make something ...,vegan,Absolutely! For a vegan breakfast yogurt that ...,True,NaN,1_35,1,PASS,The recipe response adheres to the vegan dieta...,HIGH,True,The recipe uses only vegan ingredients such as...,PASS,True
21,I'm vegan but I really want to make something ...,vegan,Absolutely! Since you're vegan and craving a h...,True,NaN,1_37,1,PASS,The recipe response adheres to the vegan dieta...,HIGH,True,The recipe uses maple syrup as a sweetener ins...,PASS,True
41,Vegetarian pizza but I don't like mushrooms or...,vegetarian,Great! Here's a delicious Vegetarian Margherit...,True,NaN,5_6,5,PASS,The recipe provided is for a Vegetarian Marghe...,HIGH,True,The recipe does not include mushrooms or bell ...,PASS,True
14,Sugar-free dessert that doesn't taste like sad...,sugar-free,Let's make a delightful **Matcha Coconut Mouss...,True,NaN,37_33,37,PASS,The recipe for Matcha Coconut Mousse adheres t...,HIGH,True,The recipe for Matcha Coconut Mousse does not ...,PASS,True
95,Halal pasta recipe - is that even a thing?,halal,Absolutely! Halal pasta recipes are very commo...,True,NaN,28_28,28,PASS,The recipe for Halal Chicken and Vegetable Pas...,HIGH,True,The recipe includes Halal chicken and optional...,PASS,True
59,I'm on a sugar-free diet but I want to make candy,sugar-free,Absolutely! Here's a delightful sugar-free rec...,True,NaN,26_29,26,PASS,The recipe for Chocolate Coconut Bites adheres...,HIGH,True,The recipe includes erythritol or monk fruit s...,PASS,True
17,Vegan protein smoothie that doesn't taste chalky,vegan,Certainly! Here's a delicious and creamy vegan...,True,NaN,19_3,19,PASS,The recipe for the Creamy Banana & Berry Vegan...,HIGH,True,"The recipe includes almond butter, which is ac...",PASS,True


In [95]:
fail_dev_df = dev[dev["label"] == "FAIL"]

In [104]:
fail_dev_df

,query,dietary_restriction,response,success,error,trace_id,query_id,label,reasoning,confidence,labeled,llm_reasoning,llm_classification,llm_classifiaction_as_bool
13,I eat pretty clean most of the time,whole30,Great! I recommend trying a **Lemon Herb Grill...,True,NaN,51_23,51,FAIL,The recipe for Lemon Herb Grilled Chicken Sala...,HIGH,True,The recipe consists of clean and fresh ingredi...,PASS,True
50,I avoid processed foods but I'm lazy,paleo,Let's go for a quick and nutritious Meal: **Ga...,True,NaN,46_19,46,FAIL,The recipe for Garlic Lemon Chickpea Salad inc...,HIGH,True,"The recipe includes a can of chickpeas, which ...",FAIL,False
23,I want something light but filling,low-carb,Certainly! I recommend trying a **Grilled Chic...,True,NaN,42_1,42,FAIL,"The recipe includes quinoa, which is a grain a...",HIGH,True,"The recipe contains grilled chicken, which is ...",FAIL,False
6,Dairy-free but cheese is okay sometimes,dairy-free,Great! I recommend trying a **Creamy Coconut C...,True,NaN,47_31,47,FAIL,The recipe for Creamy Coconut Curry with Veget...,HIGH,True,The recipe is primarily dairy-free and include...,PASS,True
62,I avoid processed foods but I'm lazy,paleo,"Great! How about a simple, wholesome Spaghetti...",True,NaN,46_3,46,FAIL,The recipe for Spaghetti Aglio e Olio does not...,HIGH,True,The recipe for Spaghetti Aglio e Olio uses sim...,PASS,True
100,I don't like vegetables but I want to eat healthy,vegetarian,Great! If you want to eat healthy while avoidi...,True,NaN,38_36,38,FAIL,The recipe provided is for 'Lemon Herb Chicken...,HIGH,True,The recipe does not include vegetables and foc...,PASS,True
24,Gluten-light recipe - I'm not celiac just sens...,gluten-free,Certainly! Here's a delicious gluten-light rec...,True,NaN,48_34,48,FAIL,The recipe provided is labeled as 'gluten-ligh...,HIGH,True,"The recipe uses boneless chicken breasts, whic...",PASS,True
86,Gluten-light recipe - I'm not celiac just sens...,gluten-free,Let's make a delicious **Lemon Herb Chicken wi...,True,NaN,48_22,48,FAIL,The recipe for Lemon Herb Chicken with Roasted...,MEDIUM,True,The recipe does not include any gluten-contain...,PASS,True
99,Gluten-light recipe - I'm not celiac just sens...,gluten-free,Certainly! Here's a delicious Gluten-Light Chi...,True,NaN,48_30,48,FAIL,"The recipe is labeled as 'gluten-light,' which...",MEDIUM,True,"The recipe includes Tamari, which is a soy sau...",PASS,True
38,Gluten-light recipe - I'm not celiac just sens...,gluten-free,Let's make a delicious **Zucchini Noodle Prima...,True,NaN,48_8,48,FAIL,The recipe for Zucchini Noodle Primavera is pr...,HIGH,True,"The recipe features Zucchini Noodle Primavera,...",PASS,True


In [97]:
TP = sum(pass_dev_df["label"] == pass_dev_df["llm_classification"])

In [98]:
FP = pass_dev_df.shape[0] - TP

In [99]:
TN = sum(fail_dev_df["label"] == fail_dev_df["llm_classification"])

In [100]:
FN = fail_dev_df.shape[0] - TN

In [101]:
print(f"TP: {TP}")
print(f"FP: {FP}")
print(f"TN: {TN}")
print(f"FN: {FN}")

TP: 24
FP: 2
TN: 5
FN: 9


In [102]:
TPR = float(TP) / (float(TP) + float(FN))
TNR = float(TN) / (float(TN) + float(FP))
print(f"TPR: {TPR}")
print(f"TNR: {TNR}")

TPR: 0.7272727272727273
TNR: 0.7142857142857143


In [94]:
true_positive_rate = dev["llm_classifiaction_as_bool"].sum() / dev["llm_classifiaction_as_bool"].count()